# Case 5 — Birleşik Risk Skoru Defteri

Case 4, dört anomali katmanını **bilinçli olarak ayrı** tuttu — tek bir katmana güvenmenin, diğer üçünün gördüğü şüpheli işlemlerin çoğunu kaçırdığını gösterdik (590.540 işlemin sadece 4'ü beş skorun hepsinde top-%1'de). Case 5 farklı bir pratik ihtiyaca cevap veriyor: gerçek bir sistemde tek bir öncelik sıralaması/karar gerekir — bu yüzden şimdi bu dört bağımsız bakış açısını, **neden** böyle birleştirdiğimizi gerekçelendirerek tek bir risk skoruna topluyoruz. İki adım, iki bölüm: önce normalizasyon, sonra (sıradaki bölümde) ağırlıklı birleştirme.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise RuntimeError("repo root not found — expected a requirements.txt somewhere above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/home/canberk/workspace/case_study')

In [2]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from src.config import settings

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

parquet_path = settings.processed_data_path / "merged_transactions.parquet"
print(f"kaynak: {parquet_path}")

kaynak: /home/canberk/workspace/case_study/data/processed/merged_transactions.parquet


## 1. Anomali Skorlarını Normalize Etme

Case 4'ün beş birincil skoru tamamen farklı ölçeklerde: `column_anomaly` 0-26 arası, `multivariate_mahalanobis` 0-8, `isolation_forest` -0,12 ile 0,25, `entity_anomaly` 0-**8028**, `temporal_anomaly` 0-**495**. Bunları doğrudan toplayıp ağırlıklandırmak anlamsız olur — en geniş ölçekli skor (entity) diğerlerini otomatik olarak ezer.

**Neden percentile-rank, neden min-max değil:** Case 4 boyunca her katmanda tek bir uç değerin ham skoru domine ettiğini gördük — `entity_anomaly`'nin en yüksek skoru (8028), sadece 2 önceki işlemden hesaplanan kararsız bir istatistikten geliyordu; `temporal_anomaly`'de de aynı mekanizma vardı (493). Min-max normalizasyon (`(x-min)/(max-min)`) tam olarak bu tek uç değere göre ölçeklenir — geri kalan her şeyi sıfıra yakın bir yere sıkıştırır. Percentile-rank (`x`'in kendi dağılımındaki yüzdelik sırası) sadece **sıralamaya** bakar, büyüklüğe değil — tek bir saçma uç değer geri kalanını bozamaz. Bu, Case 1-4'ün MAD tabanlı yöntemlere geçme sebebiyle birebir aynı mantık, `src/services/anomaly/normalization.py` ikisini yan yana hesaplayıp bunu somut gösteriyor.

In [3]:
from src.services.anomaly.combined import compute_all_anomaly_scores, PRIMARY_SCORE_COLUMNS
from src.services.anomaly.normalization import normalize_scores

all_scores = compute_all_anomaly_scores(parquet_path)
normalized = normalize_scores(all_scores, PRIMARY_SCORE_COLUMNS)
print(f"şekil: {normalized.shape}")
normalized[["TransactionID"] + PRIMARY_SCORE_COLUMNS].head(5)

şekil: (590540, 29)


,TransactionID,column_anomaly_mean_abs_zscore,multivariate_mahalanobis_distance,multivariate_isolation_forest_score,entity_anomaly_score,temporal_anomaly_score
0,2987000,0.226486,1.092386,-0.109392,0.0,0.375454
1,2987001,0.970291,1.358261,-0.095853,0.0,0.375454
2,2987002,0.289373,1.147110,-0.115453,0.0,0.742433
3,2987003,0.741222,1.769160,-0.052951,0.0,0.480542
4,2987004,0.688581,0.996790,-0.115155,0.0,0.471785


### Somut kanıt: min-max, `entity_anomaly` ve `temporal_anomaly`'i pratik olarak sıfıra sıkıştırıyor

In [4]:
comparison = pd.DataFrame({
    col: {
        "min-max IQR (25%-75%)": f"{normalized[f'{col}_minmax_normalized'].quantile(0.25):.5f} - {normalized[f'{col}_minmax_normalized'].quantile(0.75):.5f}",
        "rank IQR (25%-75%)": f"{normalized[f'{col}_rank_normalized'].quantile(0.25):.3f} - {normalized[f'{col}_rank_normalized'].quantile(0.75):.3f}",
    }
    for col in PRIMARY_SCORE_COLUMNS
}).T
comparison

,min-max IQR (25%-75%),rank IQR (25%-75%)
column_anomaly_mean_abs_zscore,0.02014 - 0.04356,0.250 - 0.750
multivariate_mahalanobis_distance,0.10541 - 0.19781,0.256 - 0.750
multivariate_isolation_forest_score,0.05082 - 0.30147,0.249 - 0.750
entity_anomaly_score,0.00003 - 0.00009,0.250 - 0.750
temporal_anomaly_score,0.00068 - 0.00261,0.250 - 0.750


`entity_anomaly_score`'da min-max'ın çeyrekler-arası aralığı **0,0000294 – 0,0000918** — pratik olarak ayırt edilemez, sıfırla aynı. `temporal_anomaly_score` da benzer şekilde (0,00068 – 0,00261). Rank-normalize'de ikisi de beklendiği gibi tam **0,25 – 0,75** veriyor — tanım gereği (her rank dağılımı düzgün/uniform). `isolation_forest_score` min-max'ta nispeten daha az bozuluyor (0,05-0,30) çünkü bu skor zaten sınırlı bir aralıkta (-0,12 ile 0,25 arası) — ama diğer dört katmanda min-max kullanmak, o katmanları birleştirilmiş skordan fiilen **silmek** anlamına gelirdi.

**Karar:** ağırlıklı birleştirmede (sıradaki bölüm) `_rank_normalized` kolonları kullanılacak.

---

**Durum:** Case 5'in 1. maddesi (normalizasyon) tamamlandı. Sıradaki madde (ağırlıklı birleştirme) sonraki bölümde eklenecek.

## 2. Weighted Aggregation

Beş rank-normalize skoru tek bir risk skoruna topluyoruz. `isFraud`'a hiç bakmadan iki ağırlıklandırma şeması hesaplayıp karşılaştırıyoruz — case brief'in "neden bu yaklaşım" sorusuna cevaben:

- **Eşit ağırlık (1/5 her biri)** — dürüst varsayılan: etiketlenmiş bir doğrulama kümemiz olmadan hangi katmana daha çok güveneceğimize dair bir kanıtımız yok, o yüzden beşini de eşit güvenilir kabul etmek en savunulabilir başlangıç noktası.
- **Redundancy-adjusted ağırlık** — Case 4/5'in kendi Spearman korelasyon matrisinden: `column_anomaly` ile iki multivariate skoru birbirleriyle güçlü korele (0,70-0,75) — çünkü multivariate'in kullandığı 3 kolon zaten column anomaly'nin ölçtüğü bilginin bir alt kümesi, aynı şeyi tekrar ölçüyorlar. `entity_anomaly` ve `temporal_anomaly` ise diğerleriyle zayıf korele (0,14-0,30) — gerçekten bağımsız bilgi taşıyorlar. Bir skor diğerleriyle ne kadar korele ise birleşik skora o kadar az **yeni** bilgi katıyor demektir; ağırlığı buna göre (korelasyonla ters orantılı) küçültüyoruz. Bu, sadece skorların birbiriyle istatistiksel ilişkisini kullanan standart bir ensemble ağırlıklandırma mantığı — etiket hiç devreye girmiyor.

In [5]:
from src.services.anomaly.aggregation import (
    compute_equal_weights,
    compute_redundancy_adjusted_weights,
    compute_final_risk_scores,
)

equal_weights = compute_equal_weights(PRIMARY_SCORE_COLUMNS)
redundancy_weights = compute_redundancy_adjusted_weights(normalized, PRIMARY_SCORE_COLUMNS)

pd.DataFrame({"eşit ağırlık": equal_weights, "redundancy-adjusted ağırlık": redundancy_weights}).round(4)

,eşit ağırlık,redundancy-adjusted ağırlık
column_anomaly_mean_abs_zscore,0.2,0.1868
multivariate_mahalanobis_distance,0.2,0.1822
multivariate_isolation_forest_score,0.2,0.1873
entity_anomaly_score,0.2,0.2199
temporal_anomaly_score,0.2,0.2238


Beklendiği gibi: `temporal_anomaly` (0,224) ve `entity_anomaly` (0,220) en yüksek ağırlığı alıyor — en bağımsız bilgiyi taşıyorlar. `column_anomaly` (0,187) ve `multivariate_mahalanobis` (0,182) en düşük — birbirleriyle ve `isolation_forest`'la örtüşen bilgi taşıyorlar.

In [6]:
final_scores = compute_final_risk_scores(normalized, PRIMARY_SCORE_COLUMNS)
print(f"şekil: {final_scores.shape}")
final_scores.head(10)

şekil: (590540, 3)


,TransactionID,final_risk_score_equal_weight,final_risk_score_redundancy_weighted
0,2987000,0.161513,0.161545
1,2987001,0.357448,0.343683
2,2987002,0.220736,0.227460
3,2987003,0.444863,0.429824
4,2987004,0.226103,0.227072
5,2987005,0.235616,0.237475
6,2987006,0.409144,0.417768
7,2987007,0.693899,0.682130
8,2987008,0.563905,0.550057
9,2987009,0.370285,0.364254


### Betimleyici kontrol — tek katmanların hiçbirinin vermediği kadar temiz bir sıralama

In [7]:
isfraud = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "isFraud"]).to_pandas()
check = final_scores.merge(isfraud, on="TransactionID")

for col in ["final_risk_score_equal_weight", "final_risk_score_redundancy_weighted"]:
    check[f"{col}_decile"] = pd.qcut(check[col], 10, labels=[str(i) for i in range(1, 11)])
    print(f"--- {col} ---")
    print((check.groupby(f"{col}_decile", observed=True)["isFraud"].mean() * 100).round(2).to_dict())

--- final_risk_score_equal_weight ---
{'1': 1.82, '2': 1.95, '3': 2.15, '4': 2.5, '5': 2.92, '6': 3.63, '7': 4.44, '8': 4.7, '9': 5.69, '10': 5.2}
--- final_risk_score_redundancy_weighted ---
{'1': 1.82, '2': 2.01, '3': 2.16, '4': 2.69, '5': 3.08, '6': 3.51, '7': 4.29, '8': 4.51, '9': 5.57, '10': 5.34}


İkisi de 1. kovadan 9. kovaya neredeyse monotonik artıyor (%1,82 → %5,6-5,7, ~3 kat) — Case 4'teki tek tek katmanların hiçbirinin (özellikle entity ve temporal'ın düzensiz örüntülerinin) veremediği kadar temiz bir sıralama. Bu, tam olarak beklenen sonuç: kusurlu ama **birbirini tamamlayan** dört sinyali birleştirmek, tek başına herhangi birinden daha güvenilir bir sıralama üretiyor — case brief'in "daha güvenilir bir anomali detection mimarisi" hedefinin doğrulanması.

İki ağırlıklandırma şeması sonuç olarak birbirine çok yakın (top-%1'in %96,7'si ortak) — hangisini seçtiğimiz kritik değil, ama redundancy-adjusted olanı öneriyoruz çünkü gerekçesi etikete değil skorların kendi istatistiksel ilişkisine dayanıyor ve tekrarlayan bilgiyi (column+multivariate) fazla ağırlıklandırmıyor.

---

**Durum:** Case 5'in her iki maddesi de tamamlandı — normalizasyon ve ağırlıklı birleştirme.

## 3. Final Raw Anomali Skoru

Önceki iki madde **mekanizmayı** kurdu (nasıl normalize edilir, nasıl ağırlıklandırılır). Bu madde, bunun **tek bir resmi çıktıya** dönüştürülmesini istiyor — şu ana kadar iki paralel aday vardı (`final_risk_score_equal_weight`, `final_risk_score_redundancy_weighted`). Bunlardan **redundancy-adjusted** olanı final olarak seçiyoruz — gerekçesi tamamen skorların kendi istatistiksel ilişkisine dayanıyor, `isFraud`'a hiç bakmıyor.

"Raw" ibaresi kasıtlı: bu, ek bir ölçekleme/kategori (düşük/orta/yüksek risk gibi) uygulanmamış, sürekli bir sayı — o kalibrasyon kararı bu skoru kullanacak sistemin işi, bu adımın değil.

In [8]:
from src.services.anomaly.aggregation import compute_final_raw_anomaly_score

final_raw = compute_final_raw_anomaly_score(normalized, PRIMARY_SCORE_COLUMNS)
print(f"şekil: {final_raw.shape}")
final_raw.describe()

şekil: (590540, 2)


,TransactionID,final_raw_anomaly_score
count,5.905400e+05,590540.000000
mean,3.282270e+06,0.500001
std,1.704744e+05,0.201231
min,2.987000e+06,0.046729
25%,3.134635e+06,0.344895
50%,3.282270e+06,0.489704
75%,3.429904e+06,0.649435
max,3.577539e+06,0.998737


### En riskli 10 işlem

In [9]:
final_raw.sort_values("final_raw_anomaly_score", ascending=False).head(10)

,TransactionID,final_raw_anomaly_score
275529,3262529,0.998737
275535,3262535,0.998616
452454,3439454,0.997415
103709,3090709,0.995837
452515,3439515,0.995120
473982,3460982,0.994960
533793,3520793,0.993348
338146,3325146,0.993074
378399,3365399,0.992737
332039,3319039,0.991745


---

## Sonuç — Case 5 ve tüm anomali tespiti hattı

**Case 4 → Case 5 zinciri:** dört bağımsız katman (column, multivariate, entity, temporal) inşa edilip doğrulandı; hiçbiri birbirine baskın gelmedi (Case 4'te top-%1 örtüşmesi sadece 4/590.540 işlemdi). Case 5 bu dört bağımsız görüşü, hangi katmanın ne kadar **yeni** bilgi taşıdığına (korelasyon matrisi) dayanarak, etikete hiç bakmadan tek bir sürekli risk skoruna topladı.

**Baştan sona hiçbir noktada `isFraud`, bir tasarım kararını yönlendirmedi** — sadece betimleyici doğrulama için kullanıldı (fraud oranının skorla birlikte arttığını göstermek gibi). Bu, Case 1'den beri korunan tek bir ilkenin sonucu.

**Nihai çıktı:** her `TransactionID` için tek bir sürekli `final_raw_anomaly_score` (`compute_final_raw_anomaly_score`) — sonraki bir aşamanın (örn. eşikleme, uyarı sistemi, ya da denetimli bir modelin girdi özelliği) doğrudan tüketebileceği şekilde. Kalıcı bir dosyaya/veritabanına yazma bu adımın kapsamında değil.